# Register Workspace Assets in an External Azure ML Registry

Share existing registered models, environments, and data assets from your workspace into a separate registry. Source assets are not modified. The destination keeps the same asset names and uses explicit, immutable versions.

**Source:** Adapted from this repository's [asset-sharing notebook](../../../notebooks/asset_sharing/sharing_assets_registries_workspaces.ipynb). The model/environment sharing flow is retained; this notebook adds data sharing, workshop configuration, preview, collision checks, and registry readback.

Run this notebook from the AML compute instance, or another machine with private connectivity and DNS for the workspace, registry, and their Storage/ACR private endpoints. Laptop IP exceptions on supporting services do not expose the private AML APIs.

Complete notebooks 02-04 first. The caller needs workspace asset access and **AzureML Registry User** on the destination. Native sharing also requires the destination registry's resource-instance rule on source Storage. The new private Terraform stack configures these permissions and managed-network outbound rules.

This registers assets, not an online endpoint or a job. Sharing an environment can trigger an image build on the workspace image-build cluster. No image is built directly in the private registry.

In [ ]:
from pathlib import Path
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential
from dotenv import dotenv_values
from azure.core.exceptions import ResourceNotFoundError
from IPython.display import display
import pandas as pd

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
for parent in (search_start, *search_start.parents):
    candidate = parent if parent.name == "workspace_readiness" else parent / "notebooks/workspace_readiness"
    if (candidate / ".env.example").is_file():
        SUITE_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run from this repository or notebooks/workspace_readiness.")

SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"

env_path = SUITE_ROOT / ".env"
if not env_path.is_file():
    raise FileNotFoundError(f"Configure {env_path} with your four Azure resource values.")
env = dotenv_values(env_path, encoding="utf-8-sig", interpolate=False)
SUBSCRIPTION_ID = (env["AZURE_SUBSCRIPTION_ID"] or "").strip()
RESOURCE_GROUP = (env["AZURE_RESOURCE_GROUP"] or "").strip()
WORKSPACE_NAME = (env["AZUREML_WORKSPACE_NAME"] or "").strip()
REGISTRY_NAME = (env["AZUREML_REGISTRY_NAME"] or "").strip()

if len(env) != 4 or not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Fill in only the four workspace and registry values in .env.")
if any(value.startswith(("<", "${")) for value in (SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Replace the .env placeholders with your Azure resource values.")

credential = AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print(f"Workspace: {WORKSPACE_NAME}; registry: {REGISTRY_NAME}")

ASSET_NAMES = {
    "data": "readiness-taxi-data",
    "environment": "readiness-taxi-environment",
    "model": "readiness-taxi-model",
}



ASSET_TYPES = ["model", "environment", "data"]
SOURCE_VERSIONS = {"model": "", "environment": "", "data": ""}
TARGET_VERSIONS = {"model": "", "environment": "", "data": ""}
ml_client_workspace = ml_client
ml_client_registry = MLClient(credential=credential, registry_name=REGISTRY_NAME)
print(f"Source workspace: {WORKSPACE_NAME}; destination registry: {REGISTRY_NAME}")


## Select Assets and Versions

The asset names and version selections are defined directly in the first cell.
A blank source version resolves the workspace latest label once; a blank target
version preserves that resolved immutable source version.

The notebook caller needs registry asset permissions in addition to private
connectivity. Network Connection Approver does not authorize asset sharing.


In [ ]:
def resolve_workspace_assets(
    workspace_client: MLClient,
    asset_types: list[str],
    settings: dict[str, str],
) -> list[dict[str, str]]:
    collections = {"model": "models", "environment": "environments", "data": "data"}
    if not asset_types or len(set(asset_types)) != len(asset_types):
        raise ValueError("Select one or more distinct asset types: model, environment, data")
    if any(kind not in collections for kind in asset_types):
        raise ValueError("Supported asset types are model, environment, and data")

    assets = []
    for kind in asset_types:
        name = settings.get(f"WORKSHOP_{kind.upper()}_NAME", "").strip()
        if not name or name.startswith("<"):
            raise ValueError(f"Set WORKSHOP_{kind.upper()}_NAME to an existing workspace asset")
        operations = {"model": workspace_client.models, "environment": workspace_client.environments, "data": workspace_client.data}[kind]
        source_version = settings.get(f"REGISTRY_{kind.upper()}_SOURCE_VERSION", "").strip()
        source = (
            operations.get(name, version=source_version)
            if source_version else operations.get(name, label="latest")
        )
        if not source.version or not source.id:
            raise ValueError(f"Workspace {kind} {name} has no registered version or resource ID")
        target_version = settings.get(f"REGISTRY_{kind.upper()}_TARGET_VERSION", "").strip() or source.version
        assets.append({
            "asset_type": kind,
            "collection": collections[kind],
            "name": name,
            "source_version": source.version,
            "target_version": target_version,
            "source_id": source.id,
        })
    return assets


def share_workspace_assets(
    workspace_client: MLClient,
    registry_client: MLClient,
    registry_name: str,
    assets: list[dict[str, str]],
) -> list[dict[str, str]]:
    if not assets:
        raise ValueError("No assets were selected for sharing")
    source_operations = {"models": workspace_client.models, "environments": workspace_client.environments, "data": workspace_client.data}
    target_operations = {"models": registry_client.models, "environments": registry_client.environments, "data": registry_client.data}
    results = []
    for asset in assets:
        operations = target_operations[asset["collection"]]
        try:
            existing = operations.get(asset["name"], version=asset["target_version"])
        except ResourceNotFoundError:
            results.append({**asset, "status": "ready", "id": ""})
        else:
            if not existing.id:
                raise RuntimeError(f"Registry readback has no resource ID for {asset['name']}:{asset['target_version']}")
            results.append({**asset, "status": "exists", "id": existing.id})
    if any(result["status"] == "exists" for result in results):
        raise ValueError("Destination asset versions already exist. Choose new TARGET_VERSIONS in the configuration cell; no assets were shared.")

    for result in results:
        operations = source_operations[result["collection"]]
        operations.share(
            name=result["name"],
            version=result["source_version"],
            registry_name=registry_name,
            share_with_name=result["name"],
            share_with_version=result["target_version"],
        )
        verified = target_operations[result["collection"]].get(
            result["name"], version=result["target_version"]
        )
        if verified.name != result["name"] or verified.version != result["target_version"] or not verified.id:
            raise RuntimeError(f"Registry readback did not match {result['name']}:{result['target_version']}")
        result["status"] = "registered"
        result["id"] = verified.id
    return results

In [ ]:
asset_settings = {
    f"WORKSHOP_{kind.upper()}_NAME": name for kind, name in ASSET_NAMES.items()
}
for kind in ASSET_NAMES:
    asset_settings[f"REGISTRY_{kind.upper()}_SOURCE_VERSION"] = SOURCE_VERSIONS[kind]
    asset_settings[f"REGISTRY_{kind.upper()}_TARGET_VERSION"] = TARGET_VERSIONS[kind]
asset_plan = resolve_workspace_assets(ml_client_workspace, ASSET_TYPES, asset_settings)
display(pd.DataFrame(asset_plan)[["asset_type", "name", "source_version", "target_version", "source_id"]])


## Share and Verify

Executing the next cell always shares the resolved `asset_plan` and retrieves
each resulting registry asset. Existing target versions stop the operation
before any writes; choose new `TARGET_VERSIONS` when necessary.

Native sharing requires registry-to-source-Storage access as well as caller RBAC.
No firewall exception or policy change is applied automatically. The integration
pipeline uses a different path: its cluster identity registers mounted model
output directly in the registry.


In [ ]:
results = share_workspace_assets(
    ml_client_workspace, ml_client_registry, REGISTRY_NAME, asset_plan
)
display(pd.DataFrame(results)[["asset_type", "name", "source_version", "target_version", "status", "id"]])
print("Every selected asset was shared and retrieved from the registry.")
